# Merge dataset com planilha Busca Fácil

**Autor:** Breno Krohling

**Data:** 11-03-2026

## Objetivo
Realizar o cruzamento de dados do dataset de coral sol a ser inserido no banco de dados em conjunto com a planilha busca fácil através de uma ordem de serviço conhecida. Com isso é possível resgatar informações que não são originais do dataset de imagens como empresa, embarcação, etc;

#### Imports

In [ ]:
import pandas as pd
from pathlib import Path
import re
import unidecode
import nltk
from nltk.corpus import stopwords
import hashlib
from PIL import Image
import imagehash
from tqdm import tqdm

#### Funções

In [ ]:
nltk.download('stopwords')
def ler_busca_facil(caminho: str) -> pd.DataFrame:
    """Realiza leitura da planilha busca fácil em um dataframe

    Args:
        caminho (str): Caminho para o arquivo da planilha busca fácil

    Returns:
        pd.DataFrame: Dataframe com a planilha busca fácil contendo dados de pacotes e arquivados.
    """
    dados_pacotes = pd.read_excel(caminho, sheet_name='Pacotes',header=[1,2], dtype={('Referências Externas', 'Ordem de Serviço'): str})
    dados_arquivados = pd.read_excel(caminho, sheet_name='Arquivados',header=[1,2], dtype={('Referências Externas', 'Ordem de Serviço'): str})
    dados = pd.concat([dados_pacotes, dados_arquivados])
    dados = dados.drop(dados.columns[0], axis=1)

    dados.columns = dados.columns.droplevel(0)
    dados = dados[["Caminho Completo", "Ordem de Serviço", "Empresa","Nome", "Tipo","Arquivo mais antigo", "Arquivo mais novo"]]

    return dados

def ler_dataset_coral_sol(pasta_base: str, separador: str = '\\') -> pd.DataFrame:
    """Realiza a leitura e manipulação inicial do dataset.

    Nota: Essa função está escrita para o dataset de coral sol rotulado por especialista. Outros datasets podem necessitar de outra função de leitura.
    Estrutura do dataset: PASTA_BASE/LABEL/OS/IMAGEM
    Args:
        pasta_base (str): caminho para a pasta base do dataset.
        separador (str) : Simbolo utilizado apra fazer o split do caminho completo do arquivo.

    Returns:
        pd.DataFrame: dataframe com as principais informações do dataset.
    """
    dados = []
    imagens = list(Path(pasta_base).rglob("*"))

    for arquivo in tqdm(imagens, total=len(imagens)):
        if arquivo.is_file():
            divisoes = str(arquivo).split(separador)
            img = Image.open(arquivo)
            img.thumbnail((640,640))
            phash = imagehash.phash(img)
            dados.append({
                'label': 'coral-sol' if divisoes[-3] == 'POSITIVO' else None,
                'ordem_servico': str(divisoes[-2]),
                'arquivo': divisoes[-1],
                'path_original': arquivo,
                'hash': hashlib.sha256(divisoes[-1].encode()).hexdigest(),
                'phash': phash 
            })
    return pd.DataFrame(dados)

def gerar_stopwords() -> tuple[list[str], list[str]]:
    """Gera lista de stopwords para pré-processamento textual

    Returns:
        tuple[list[str], list[str]]: Lista de stopwords comuns da lingua portuguesa e lista de stopwrods customizadas para o contexto
    """
    stopwords_pt = set(stopwords.words('portuguese'))
    stopwords_customizadas = [
        'não', 'nao', 'que', 'metro', 'linear', 'maior', 'menor', 'evento' 
    ]

    return stopwords_pt, stopwords_customizadas

def normalizar(texto: str) -> str:
    """Normaliza um texto com padrão lower case e unicode.

    Args:
        texto (str): Texto para aplicar normalização

    Returns:
        str: Texto normalizado
    """
    texto = texto.lower()
    return unidecode.unidecode(texto)

def split_camel_case(texto: str) -> str:
    """Separa palavras que estão em camel case

    Args:
        texto (str): texto a ser processado.

    Returns:
        str: Texto processado.
    """
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', texto)

def tokenizar(texto: str) -> list[str]:
    """Tokeniza um texto de entrada.

    Args:
        texto (str): Texto a ser tokenizado.

    Returns:
        list[str]: Lista de tokens obtidos a partir do texto
    """
    texto = split_camel_case(texto)
    texto = re.sub(r"[^\w\s]", "", texto) # remove pontuação
    tokens =  re.split(r'[_\-\s\.]+', texto)
    tokens = [re.sub(r"[\d+]", "", t) for t in tokens]
    return tokens

def limpar_tokens(tokens: list[str], stopwords_pt: list[str], stopwords_customizadas: list[str]) -> list[str]:
    """Realiza uma limpeza de tokens a partir de regas bem defnidas.

        Remove:
            - Pavaras com tamanho inferior a três
            - Número
            - Stopwrods da lingua portuguesa
            - Stopwrods do contexto
    Args:
        tokens (list[str]): Lista de tokens a ser processada
        stopwords_pt (list[str]): Stopwords lingua portuguesa
        stopwords_customizadas (list[str]): stopwords contexto

    Returns:
        list[str]: Lista de tokens filtrados
    """
    resultado = []
    for t in tokens:
        if len(t) < 3:
            continue
        if t.isdigit():
            continue
        if t in stopwords_pt:
            continue
        if t in stopwords_customizadas:
            continue
        
        resultado.append(t)
    return resultado

def normalizar_plural(tokens: list[str]) -> list[str]:
    """Normaliza tokens no plural

    Args:
        tokens (list[str]): Lista de tokens a ser normalizada

    Returns:
        list[str]: Tokens normalizados
    """
    resultado = []
    for t in tokens:
        if t.endswith('s') and len(t) > 4:
            t = t[:-1]
        resultado.append(t)
          
    return resultado

def processsar_arquivo(nome_arquivo: str) -> list[str]:
    """Realiza o processamento do nome do arquivo para extrair tokens.

    Args:
        nome_arquivo (str): Nome do arquivo de interesse

    Returns:
        list[str]: _description_Lista de tokens extraídos a partir do nome do arquivo
    """
    nome = normalizar(Path(nome_arquivo).stem)
    tokens = tokenizar(nome)
    stopwords_pt, stopwords_customizadas = gerar_stopwords()
    tokens = limpar_tokens(tokens, stopwords_pt, stopwords_customizadas)  
    tokens = normalizar_plural(tokens)
    return tokens

def tokens2classes(tokens: list[str]) -> list[str]:
    """Mapea tokens em classes do contexto.

    As classes finais utilizadas foram obtidos após análise do tokens gerados a partir dos nomes dos arquivos e imagem. 

    Args:
        tokens (list[str]): Lista de tokens a serem tansformados em classes

    Returns:
        list[str]: Lista de classes
    """
    classes = [
        "anodo",
        "colarancoragem",
        "colaranodo",
        "colarbatente",
        "enrijecedor",
        "duto",
        "flutuador",
        "cabo",
        "corrosao",
        "assoreamento",
        "dano",
        "reparo",
        "folga",
        "incrustacao",
        "rompimentostrake",
        "deslizestrake",
        "sucata"
    ]
    classes_selecionadas = []
    for t in tokens:
        for c in classes:
            if c in t:
                classes_selecionadas.append(c)
    return classes_selecionadas

def adiciona_coral_se_flag(l: list[str], flag: str) -> list[str]:
    """Concatena coralsol a lista de classes caso flag seja coral-sol

    Args:
        l (list[str]): lsita de classes já selecionada
        flag (str): flag indicando label orginal

    Returns:
        list[str]: lista com classes atualizadas
    """
    return l + ['coralsol'] if flag == 'coral-sol' else l


#### Constantes

In [ ]:
CAMINHO_BUSCA_FACIL = '../Pacotes.xlsx'
CAMINHO_DATASET = '../coral_sol_especialista'

#### Geração de classes a partir de descrição

In [ ]:
dados_busca_facil = ler_busca_facil(CAMINHO_BUSCA_FACIL)
dados_dataset = ler_dataset_coral_sol(CAMINHO_DATASET)
dados_dataset['tokens'] = dados_dataset['arquivo'].apply(processsar_arquivo)
dados_dataset['classes'] = dados_dataset['tokens'].apply(tokens2classes)
dados_dataset["classes"] = dados_dataset.apply(
    lambda row: adiciona_coral_se_flag(row.classes, row.label), axis=1
)

In [ ]:
dados_cruzados = dados_dataset.merge(dados_busca_facil, left_on='ordem_servico', right_on='Ordem de Serviço', how="left")
dados_cruzados['Empresa'] = dados_cruzados['Empresa'].fillna('UNKOWN')
dados_cruzados['Nome'] = dados_cruzados['Nome'].fillna('UNKOWN')
dados_cruzados['Tipo'] = dados_cruzados['Tipo'].fillna('UNKOWN')
dados_cruzados['Caminho Completo'] = dados_cruzados['Caminho Completo'].fillna('UNKOWN')
dados_cruzados = dados_cruzados.drop(columns=['label', 'tokens', 'Ordem de Serviço'])
dados_cruzados = dados_cruzados.drop_duplicates(subset='phash', keep='first')
dados_cruzados = dados_cruzados.drop_duplicates(subset='hash', keep='first')
dados_cruzados.to_csv('dados_dataset_coral_sol.csv')